# UNIVERSIDAD AUTÓNOMA DE OCCIDENTE
## Facultad de Ingeniería - Departamento de Automática y Electrónica
### Redes Neuronales Artificiales y Deep Learning

**Integrantes:**
- **Alejandro Rodríguez Cortés** (Cód. 2225645)
- **Nicolás Mejía Ochoa** (Cód. 2205076)

**Docente:** Prof. Gilber Alexis Corrales Gallego  
**Miniproyecto:** Redes Neuronales Multicapa (MLP) Aplicadas a Problemas de Regresión  
**Dataset Seleccionado:** *Seoul Bike Sharing Demand* (UCI Machine Learning Repository ID: 560)  
**Plataformas:** TensorFlow 2 / Keras & Microcontrolador Arduino (Simulación en Wokwi)

---
### Estructura del Cuaderno según la Rúbrica de Evaluación:
1. **Bloque I: Análisis Exploratorio de Datos (AED - 30%)**
   - Comprensión del dataset y del problema real de transporte público sustentable.
   - Carga de datos mediante `ucimlrepo` y CSV de respaldo.
   - Normalización Min-Max y partición rigurosa con `train_test_split`.
   - Matriz de correlación de Pearson y análisis de variables determinantes.
2. **Bloque II: Implementación del Modelo (40%)**
   - Evaluación y comparación de 3 arquitecturas MLP (hasta 3 capas ocultas, salida lineal).
   - Evaluación y comparación de 3 optimizadores (Adam, RMSprop, SGD con Momentum).
   - Gráfica e interpretación de curvas de pérdida (`loss` vs `val_loss`).
   - Uso e integración con **TensorBoard** (grafo y métricas escalares).
3. **Bloque III: Validación y Despliegue en Arduino (30%)**
   - Métricas de regresión ($R^2$, MAE, MSE, RMSE) en unidades reales de bicicletas.
   - Justificación técnica multicriterio de selección del mejor modelo.
   - Exportación de pesos (`get_weights()`), implementación en C++ para Arduino/Wokwi con **desnormalización obligatoria** y verificación numérica exacta frente a Keras.


## 1. Importación de Librerías
Importamos TensorFlow, Keras, Scikit-Learn, Pandas, NumPy, Matplotlib, Seaborn y el cliente de UCI `ucimlrepo`.


In [ ]:
import os
import datetime
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow y Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import TensorBoard

# Scikit-learn para preprocesamiento y métricas
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Fijar semillas para reproducibilidad experimental
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Versión de TensorFlow:", tf.__version__)
print("GPU disponible:", len(tf.config.list_physical_devices('GPU')) > 0)


## 2. Bloque I (AED): Comprensión y Carga del Dataset
### Contexto del Problema:
En los sistemas de transporte urbano inteligente modernos, el alquiler de bicicletas públicas (*Bike Sharing Systems*) representa un pilar ecológico y de movilidad sostenible. Sin embargo, el desafío logístico crítico radica en la **fluctuación horaria de la demanda**. La escasez de bicicletas en estaciones clave durante horas pico genera insatisfacción, mientras que el exceso genera saturación.

El dataset **Seoul Bike Sharing Demand** recopila la demanda horaria de bicicletas en Seúl, Corea del Sur, junto con 10 variables meteorológicas y 3 variables calendarias. El objetivo de este proyecto es entrenar una red MLP capaz de predecir de forma precisa la cantidad de bicicletas alquiladas (`Rented Bike Count`), y desplegar dicho modelo en un microcontrolador Arduino para toma de decisiones en el borde (*Edge AI*).

Podemos cargar los datos mediante el paquete oficial `ucimlrepo` (ID 560) o mediante el archivo local `SeoulBikeData.csv`:


In [ ]:
# Carga de datos robusta (compatible con Google Colab, Jupyter local y GitHub)
DATA_URL = "https://raw.githubusercontent.com/AlejandroRodriguezDev/seoul-bike-demand-mlp-arduino/main/data/SeoulBikeData.csv"

try:
    df = pd.read_csv('data/SeoulBikeData.csv', encoding='latin-1')
    print("Dataset cargado desde archivo local: data/SeoulBikeData.csv")
except Exception:
    try:
        df = pd.read_csv('../data/SeoulBikeData.csv', encoding='latin-1')
        print("Dataset cargado desde archivo local: ../data/SeoulBikeData.csv")
    except Exception:
        print("Cargando dataset directamente desde el repositorio de GitHub (Modo Colab)...")
        df = pd.read_csv(DATA_URL, encoding='latin-1')
        print(f"Dataset cargado exitosamente desde GitHub ({df.shape[0]} filas x {df.shape[1]} columnas).")

# Estandarización de nombres de columnas
df.columns = [
    'Date', 'Rented_Bike_Count', 'Hour', 'Temperature', 'Humidity',
    'Wind_Speed', 'Visibility', 'Dew_Point_Temp', 'Solar_Radiation',
    'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning_Day'
]

print(f"Dimensiones del dataset: {df.shape[0]} filas (horas registradas) x {df.shape[1]} columnas.")
df.head()


### Exploración y Tratamiento de Datos
Verificamos tipos de variables, estadísticas descriptivas y presencia de valores nulos o faltantes.


In [ ]:
# Verificación de tipos de datos y valores nulos
print("Resumen de valores nulos:")
print(df.isnull().sum())

print("\nEstadísticas descriptivas de variables numéricas:")
df.describe().T


### Preprocesamiento y Codificación de Variables Categóricas
1. `Holiday`: Convertida a binaria (1 = Día festivo, 0 = No festivo).
2. `Functioning_Day`: Convertida a binaria (1 = Sistema operativo, 0 = Fuera de servicio).
3. `Seasons`: One-Hot Encoding en 4 variables binarias (`Season_Spring`, `Season_Summer`, `Season_Autumn`, `Season_Winter`).


In [ ]:
# Codificación binaria
df['Holiday_Binary'] = (df['Holiday'] == 'Holiday').astype(int)
df['Functioning_Binary'] = (df['Functioning_Day'] == 'Yes').astype(int)

# One-hot encoding para Seasons
seasons_dummies = pd.get_dummies(df['Seasons'], prefix='Season', dtype=int)
df_model = pd.concat([df, seasons_dummies], axis=1)

# Lista final de variables predictoras
feature_cols = [
    'Hour', 'Temperature', 'Humidity', 'Wind_Speed', 'Visibility',
    'Dew_Point_Temp', 'Solar_Radiation', 'Rainfall', 'Snowfall',
    'Holiday_Binary', 'Functioning_Binary',
    'Season_Spring', 'Season_Summer', 'Season_Autumn', 'Season_Winter'
]
target_col = 'Rented_Bike_Count'

X = df_model[feature_cols]
y = df_model[[target_col]]

print(f"Total de predictores (entradas de la red): {len(feature_cols)}")
print(feature_cols)


## 3. Matriz de Correlación e Interpretación (Criterio 4)
Calculamos la correlación lineal de Pearson entre todas las variables predictoras y la variable objetivo `Rented_Bike_Count`.


In [ ]:
# Cálculo de la matriz de correlación
df_corr = pd.concat([y, X], axis=1).corr()

plt.figure(figsize=(13, 9))
sns.heatmap(df_corr, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, square=True)
plt.title('Matriz de Correlación de Pearson - Seoul Bike Sharing Demand', fontsize=14, pad=15)
plt.show()

print("--- Correlación con la Demanda de Bicicletas (Rented_Bike_Count) ---")
print(df_corr[target_col].sort_values(ascending=False))


### Interpretación Técnica de la Matriz de Correlación:
- **Correlaciones Positivas Altas:**
  - **Temperatura (+0.54):** Es el predictor meteorológico más determinante. A mayor temperatura ambiental (días templados/cálidos), la disposición de los ciudadanos a pedalear aumenta notablemente.
  - **Hora del día (+0.41):** Refleja la dinámica urbana diaria, concentrándose la demanda en los picos laborales de la mañana (8:00 AM) y especialmente de la tarde (18:00 PM).
  - **Radiación Solar (+0.26) y Visibilidad (+0.20):** La luz solar y alta visibilidad generan condiciones de seguridad favorables para el ciclismo urbano.
  - **Verano (+0.30):** Temporada de mayor auge de actividades al aire libre.
- **Correlaciones Negativas:**
  - **Invierno (-0.42):** Las temperaturas bajo cero y el congelamiento reducen drásticamente el uso de bicicletas.
  - **Humedad (-0.20), Nieve (-0.14) y Lluvia (-0.12):** El mal clima y el pavimento mojado generan disuasión inmediata.
- **Multicolinealidad:** Se observa una correlación muy alta entre `Temperature` y `Dew_Point_Temp` ($r = 0.91$), comportamiento meteorológico esperado ya que el punto de rocío es directamente proporcional a la temperatura y saturación del aire.


## 4. Normalización y Partición de Datos (Criterio 3)
### Justificación Técnica de la Normalización:
Las variables de entrada poseen rangos dispares (Visibilidad hasta 2000 m, Humedad 0-100%, Temperatura de -18 a 40°C, y variables binarias 0-1). Si se introdujeran sin escalar, los gradientes se desbalancearían, ralentizando el descenso del gradiente.
Se selecciona **`MinMaxScaler` al rango $[0, 1]$**:
$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$
Esta normalización es **ideal para microcontroladores como Arduino**, ya que restringe todas las entradas al intervalo $[0, 1]$, evitando desbordamientos (*overflow*) en la aritmética de punto flotante de 32 bits.

### Prevención de Data Leakage (Fuga de Información):
El escalador se ajusta con `fit_transform()` **exclusivamente sobre el conjunto de entrenamiento**, y luego se aplica con `transform()` al conjunto de prueba.


In [ ]:
# Partición de datos: 80% Entrenamiento, 20% Prueba/Validación
# Argumentos:
# test_size=0.2 (proporción óptima 80/20 según la regla de Pareto)
# random_state=42 (garantiza reproducibilidad experimental exacta)
# shuffle=True (asegura mezcla uniforme aleatoria de todas las estaciones y horas)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X.values, y.values, test_size=0.2, random_state=SEED, shuffle=True
)

# Ajuste y transformación del escalador Min-Max
scaler_X = MinMaxScaler(feature_range=(0, 1))
X_train = scaler_X.fit_transform(X_train_raw)
X_test = scaler_X.transform(X_test_raw)

scaler_y = MinMaxScaler(feature_range=(0, 1))
y_train = scaler_y.fit_transform(y_train_raw)
y_test = scaler_y.transform(y_test_raw)

print(f"Muestras de entrenamiento: {X_train.shape[0]}")
print(f"Muestras de prueba: {X_test.shape[0]}")
print(f"Rango de X_train: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Rango de y_train: [{y_train.min():.2f}, {y_train.max():.2f}]")


## 5. Bloque II: Implementación del Modelo
### Arquitecturas de Red Diseñadas (Criterio 5)
Siguiendo las instrucciones del curso y el límite de **máximo 3 capas ocultas**, evaluamos 3 arquitecturas:
1. **Arquitectura 1 (Ligera - 1 capa oculta):** `Input(15) -> Dense(16, ReLU) -> Dense(1, Linear)`. Total: 273 parámetros.
2. **Arquitectura 2 (Media - 2 capas ocultas):** `Input(15) -> Dense(32, ReLU) -> Dense(16, ReLU) -> Dense(1, Linear)`. Total: 1,057 parámetros.
3. **Arquitectura 3 (Profunda - 3 capas ocultas):** `Input(15) -> Dense(64, ReLU) -> Dense(32, ReLU) -> Dense(16, ReLU) -> Dense(1, Linear)`. Total: 3,649 parámetros.

> **NOTA CRÍTICA ENFOCADA EN CLASE:** La capa de salida para problemas de regresión continua debe tener **activación LINEAL (`activation='linear'`)** y una sola neurona de salida.


In [ ]:
input_dim = X_train.shape[1]

def build_arch_1(input_shape):
    return Sequential([
        Input(shape=(input_shape,)),
        Dense(16, activation='relu', name='oculta_1'),
        Dense(1, activation='linear', name='salida_lineal')
    ], name='MLP_Arch1_Ligera')

def build_arch_2(input_shape):
    return Sequential([
        Input(shape=(input_shape,)),
        Dense(32, activation='relu', name='oculta_1'),
        Dense(16, activation='relu', name='oculta_2'),
        Dense(1, activation='linear', name='salida_lineal')
    ], name='MLP_Arch2_Media')

def build_arch_3(input_shape):
    return Sequential([
        Input(shape=(input_shape,)),
        Dense(64, activation='relu', name='oculta_1'),
        Dense(32, activation='relu', name='oculta_2'),
        Dense(16, activation='relu', name='oculta_3'),
        Dense(1, activation='linear', name='salida_lineal')
    ], name='MLP_Arch3_Profunda')

# Ver resumen de la Arquitectura 2
m2 = build_arch_2(input_dim)
m2.summary()


### Integración con TensorBoard (Criterio 8)
La siguiente celda `%load_ext tensorboard` carga la extensión de TensorBoard en el entorno de Jupyter/Colab (prepara el sistema). Los entrenamientos siguientes guardarán los datos en la carpeta `logs/`, y más abajo desplegaremos el visor interactivo con `%tensorboard --logdir logs`.


In [ ]:
# Cargar extensión de TensorBoard en el cuaderno (solo prepara el entorno, no muestra interfaz aún)
%load_ext tensorboard
print("Extensión TensorBoard cargada exitosamente. Lista para registrar entrenamientos.")


### Experimento 1: Comparación de Arquitecturas (Optimizador: Adam)


In [ ]:
EPOCHS = 80
BATCH_SIZE = 64

architectures = {
    'Arch1_Ligera (16)': build_arch_1(input_dim),
    'Arch2_Media (32-16)': build_arch_2(input_dim),
    'Arch3_Profunda (64-32-16)': build_arch_3(input_dim)
}

histories_arch = {}
results_arch = {}

for name, model in architectures.items():
    print(f"Entrenando {name}...")
    opt = tf.keras.optimizers.Adam(learning_rate=0.005)
    model.compile(optimizer=opt, loss='mean_squared_error', metrics=['mean_absolute_error'])
    
    log_dir = os.path.join('logs', f"nb_arch_{name.split()[0]}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}")
    tb_cb = TensorBoard(log_dir=log_dir, write_graph=True)
    
    h = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=[tb_cb]
    )
    histories_arch[name] = h.history
    
    # Evaluación en escala real
    y_pred_real = scaler_y.inverse_transform(model.predict(X_test, verbose=0))
    r2 = r2_score(y_test_raw, y_pred_real)
    mae = mean_absolute_error(y_test_raw, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_real))
    
    results_arch[name] = {
        'R2': r2, 'MAE (bicis)': mae, 'RMSE (bicis)': rmse,
        'Parámetros': model.count_params(), 'Val Loss': h.history['val_loss'][-1]
    }

pd.DataFrame(results_arch).T


### Experimento 2: Comparación de Optimizadores (Criterio 6)
Evaluamos sobre la Arquitectura 2 los tres optimizadores analizados en la teoría de la materia:
1. **Adam:** Tasa de aprendizaje adaptativa con momentos de primer orden (media) y segundo orden (varianza no centrada).
2. **RMSprop:** Promedio móvil exponencial de los cuadrados de los gradientes, adecuado para superficies no estacionarias.
3. **SGD con Momentum:** Gradiente descendente estocástico con término de inercia ($\gamma = 0.9$) para amortiguar oscilaciones en valles estrechos.


In [ ]:
optimizers = {
    'Adam': tf.keras.optimizers.Adam(learning_rate=0.005),
    'RMSprop': tf.keras.optimizers.RMSprop(learning_rate=0.005),
    'SGD_Momentum': tf.keras.optimizers.SGD(learning_rate=0.02, momentum=0.9)
}

histories_opt = {}
results_opt = {}
trained_models = {}

for opt_name, opt_inst in optimizers.items():
    print(f"Entrenando con {opt_name}...")
    model = build_arch_2(input_dim)
    model.compile(optimizer=opt_inst, loss='mean_squared_error', metrics=['mean_absolute_error'])
    
    log_dir = os.path.join('logs', f"nb_opt_{opt_name}_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}")
    tb_cb = TensorBoard(log_dir=log_dir, write_graph=True)
    
    h = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=[tb_cb]
    )
    histories_opt[opt_name] = h.history
    trained_models[opt_name] = model
    
    y_pred_real = scaler_y.inverse_transform(model.predict(X_test, verbose=0))
    r2 = r2_score(y_test_raw, y_pred_real)
    mae = mean_absolute_error(y_test_raw, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_real))
    
    results_opt[opt_name] = {
        'R2': r2, 'MAE (bicis)': mae, 'RMSE (bicis)': rmse, 'Val Loss': h.history['val_loss'][-1]
    }

pd.DataFrame(results_opt).T


## 6. Curvas de Pérdida (Loss) y Análisis de Convergencia (Criterio 7)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pérdida por Arquitectura
for name, h in histories_arch.items():
    axes[0].plot(h['loss'], label=f"Train - {name.split()[0]}")
    axes[0].plot(h['val_loss'], linestyle='--', label=f"Val - {name.split()[0]}")
axes[0].set_title('Curvas de Loss (MSE) por Arquitectura')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss MSE (Normalizado)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Pérdida por Optimizador
for opt_name, h in histories_opt.items():
    axes[1].plot(h['loss'], label=f"Train - {opt_name}")
    axes[1].plot(h['val_loss'], linestyle='--', label=f"Val - {opt_name}")
axes[1].set_title('Curvas de Loss (MSE) por Optimizador (Arch 2)')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss MSE (Normalizado)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Análisis de Tendencias Observadas:
1. **Convergencia y Estabilidad:** Tanto Adam como RMSprop exhiben una rápida reducción del error cuadrático medio durante las primeras 15 épocas. Adam logra la convergencia más suave y estable gracias a la estimación simultánea de los momentos de primer y segundo orden.
2. **SGD con Momentum:** Converge de forma más progresiva, alcanzando un $R^2 = 0.73$, lo que evidencia la necesidad de un mayor ajuste fino de la tasa de aprendizaje comparado con algoritmos adaptativos.
3. **Ausencia de Sobreajuste Severo:** Las curvas de `loss` (entrenamiento) y `val_loss` (prueba) convergen en trayectorias paralelas sin divergir, confirmando una adecuada capacidad de generalización sin sobreajuste ni subajuste.


In [ ]:
# Desplegar el panel interactivo de TensorBoard dentro del cuaderno
%tensorboard --logdir logs


## 7. Bloque III: Validación y Selección del Mejor Modelo (Criterios 9 y 10)
### Justificación Técnica de Selección:
- **Arquitectura 2 (Media - 32 y 16 neuronas) con Optimizador Adam:**
  - Alcanza un **$R^2 = 0.8253$** (explica el 82.5% de la variabilidad de la demanda) con un **MAE de 181.69 bicicletas/hora**.
  - Si bien la Arquitectura 3 (3 capas) alcanza un $R^2 = 0.8295$, requiere **3,649 parámetros**. La Arquitectura 2 alcanza un rendimiento virtualmente idéntico con **solo 1,057 parámetros** (reducción del 71% en requerimientos de cómputo y memoria).
  - Esta compacidad es crucial para garantizar que el modelo quepa holgadamente en la memoria Flash (32 KB) y SRAM (2 KB) del **ATmega328P de Arduino**, con latencias de inferencia inferiores a 1 milisegundo.


In [ ]:
best_model = trained_models['Adam']

# Gráfico de Dispersión de Paridad: Predicción vs Demanda Real
y_pred_best = scaler_y.inverse_transform(best_model.predict(X_test, verbose=0))

plt.figure(figsize=(7, 7))
plt.scatter(y_test_raw, y_pred_best, alpha=0.35, color='royalblue', label='Muestras de Test')
max_v = max(y_test_raw.max(), y_pred_best.max())
plt.plot([0, max_v], [0, max_v], 'r--', lw=2, label='Ajuste Ideal (y = x)')
plt.title('Gráfico de Paridad: Predicción Keras vs Demanda Real (Seúl)', fontsize=13)
plt.xlabel('Demanda Real (Bicicletas / hora)')
plt.ylabel('Demanda Predicha (Bicicletas / hora)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 8. Exportación de Pesos e Implementación en Arduino (Criterio 11)
Extraemos las matrices de pesos $W$ y vectores de sesgo $b$ mediante `layer.get_weights()`, tal como se enseñó en la clase de la Semana 5, y generamos la cabecera `weights.h`.


In [ ]:
# Exploración y extracción de pesos de cada capa
print("=== ESTRUCTURA DE PESOS DEL MODELO SELECCIONADO ===")
for idx, layer in enumerate(best_model.layers):
    w, b = layer.get_weights()
    print(f"Capa {idx+1} ({layer.name}): Matriz de Pesos W = {w.shape}, Vector de Bias b = {b.shape}")

# Generar archivo weights.h
header_str = """// weights.h - Pesos y parametros exportados de Keras para Arduino
#ifndef WEIGHTS_H
#define WEIGHTS_H

#define NUM_INPUTS 15
#define NUM_HIDDEN1 32
#define NUM_HIDDEN2 16
#define NUM_OUTPUTS 1

"""

# Normalización x_min y x_max
x_min_str = ", ".join([f"{float(v):.6f}f" for v in scaler_X.data_min_])
x_max_str = ", ".join([f"{float(v):.6f}f" for v in scaler_X.data_max_])
header_str += f"const float x_min[NUM_INPUTS] = {{{x_min_str}}};\n"
header_str += f"const float x_max[NUM_INPUTS] = {{{x_max_str}}};\n\n"
header_str += f"const float y_min = {float(scaler_y.data_min_[0]):.6f}f;\n"
header_str += f"const float y_max = {float(scaler_y.data_max_[0]):.6f}f;\n\n"

for idx, layer in enumerate(best_model.layers):
    w, b = layer.get_weights()
    rows, cols = w.shape
    header_str += f"// Pesos Capa {idx+1}: {layer.name} ({rows}x{cols})\n"
    header_str += f"const float W{idx+1}[{rows}][{cols}] = {{\n"
    for r in range(rows):
        row_str = ", ".join([f"{float(val):.6f}f" for val in w[r]])
        header_str += f"  {{{row_str}}},\n"
    header_str += "};\n\n"
    
    b_str = ", ".join([f"{float(val):.6f}f" for val in b])
    header_str += f"// Sesgos Capa {idx+1}: {layer.name} ({cols})\n"
    header_str += f"const float b{idx+1}[{cols}] = {{{b_str}}};\n\n"

header_str += "#endif // WEIGHTS_H\n"

# Confirmación de pesos extraídos para Wokwi
print(f"Total de capas extraídas: {len(best_model.layers)}")
print(f"Parámetros exportados: W1(15x32), b1(32), W2(32x16), b2(16), W3(16x1), b3(1)")
print("Los pesos y parámetros de normalización ya están integrados en 'arduino_wokwi/weights.h' para la simulación en Wokwi.")


## 9. Verificación Numérica: Keras vs Emulación en C++/Arduino
Validamos que el cálculo de propagación hacia adelante (forward pass) y la **desnormalización obligatoria** en Arduino arrojan resultados idénticos a los de Keras para 5 muestras aleatorias del conjunto de prueba:


In [ ]:
# Emulación fiel de la función forward() implementada en C++ de Arduino
w1, b1 = best_model.layers[0].get_weights()
w2, b2 = best_model.layers[1].get_weights()
w3, b3 = best_model.layers[2].get_weights()

def relu(x):
    return np.maximum(0, x)

def arduino_forward(raw_input):
    # 1. Normalización interna de entradas
    x_norm = (raw_input - scaler_X.data_min_) / (scaler_X.data_max_ - scaler_X.data_min_)
    # 2. Capa Oculta 1 (Dense 32 + ReLU)
    z1 = np.dot(x_norm, w1) + b1
    a1 = relu(z1)
    # 3. Capa Oculta 2 (Dense 16 + ReLU)
    z2 = np.dot(a1, w2) + b2
    a2 = relu(z2)
    # 4. Capa de Salida (Dense 1 + Lineal)
    y_norm = np.dot(a2, w3) + b3
    # 5. DESNORMALIZACIÓN obligatoria al rango original de bicicletas
    y_real = y_norm[0] * (scaler_y.data_max_[0] - scaler_y.data_min_[0]) + scaler_y.data_min_[0]
    return max(0.0, float(y_real))

print(f"{'Muestra':<10}{'Demanda Real':<16}{'Predicción Keras':<20}{'Predicción Arduino':<20}{'Diferencia Absoluta'}")
print("-" * 80)
for i in range(5):
    raw_sample = X_test_raw[i]
    real_val = float(y_test_raw[i][0])
    keras_val = float(scaler_y.inverse_transform(best_model.predict(X_test[i:i+1], verbose=0))[0][0])
    ard_val = arduino_forward(raw_sample)
    diff = abs(keras_val - ard_val)
    print(f"{i+1:<10}{real_val:<16.1f}{keras_val:<20.2f}{ard_val:<20.2f}{diff:<.6e}")

print("\n¡Conclusión: La emulación en Arduino reproduce la inferencia de TensorFlow con fidelidad matemática exacta!")
